In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# ====================== 配置项（请修改为实际路径） ======================
# 含学科数据文件路径
FILE_WITH_SUBJECT = r"C:\Users\33759\Desktop\陈正扬\处理后数据（含学科）\合并数据（含学科）.xlsx"
# 不含学科数据文件路径
FILE_WITHOUT_SUBJECT = r"C:\Users\33759\Desktop\陈正扬\处理后数据（不含学科）\合并数据（不含学科）.xlsx"

# Jupyter Notebook命令：确保图表内嵌显示
%matplotlib inline
# 设置中文字体（解决matplotlib中文乱码问题）
plt.rcParams["font.family"] = ["SimHei", "Microsoft YaHei"]
plt.rcParams["axes.unicode_minus"] = False  # 解决负号显示问题

# 985高校名单
TOP985_UNIVERSITIES = [
    "清华大学", "北京大学", "中国人民大学", "北京航空航天大学", "北京理工大学", "中国农业大学", "北京师范大学", "中央民族大学", "南开大学", "天津大学", "大连理工大学", "东北大学", "吉林大学", "哈尔滨工业大学", "复旦大学", "同济大学", "上海交通大学", "华东师范大学", "南京大学", "东南大学", "浙江大学", "中国科学技术大学", "厦门大学", "山东大学", "中国海洋大学", "武汉大学", "华中科技大学", "湖南大学", "中南大学", "国防科学技术大学", "中山大学", "华南理工大学", "四川大学", "电子科技大学", "重庆大学", "西安交通大学", "西北工业大学", "西北农林科技大学", "兰州大学"
]

含学科数据可视化-堆叠面积图

In [ ]:
# ====================== 含学科数据：立项数量-年份-学科门类堆叠面积图 ======================
print("=== 绘制堆叠面积图 ===")
# 读取数据
df_with_subject = pd.read_excel(FILE_WITH_SUBJECT)
# 数据预处理
df_with_subject = df_with_subject.dropna(subset=["立项年份", "学科门类"])
df_with_subject["立项年份"] = df_with_subject["立项年份"].astype(str)

# 按年份+学科门类统计立项数
area_data = df_with_subject.groupby(["立项年份", "学科门类"])["项目名称"].count().reset_index()
area_data = area_data.pivot(index="立项年份", columns="学科门类", values="项目名称").fillna(0)

# 绘制堆叠面积图
plt.figure(figsize=(12, 6))
area_data.plot.area(stacked=True, alpha=0.8, ax=plt.gca(), colormap="tab20")
plt.title("2020-2025年各学科门类立项数量堆叠面积图", fontsize=14, pad=20)
plt.xlabel("立项年份", fontsize=12)
plt.ylabel("立项数量", fontsize=12)
plt.legend(title="学科门类", bbox_to_anchor=(1.05, 1), loc="upper left")  # 图例靠右
plt.tight_layout()
# 保存图片（可替换）
plt.savefig(r"C:\Users\33759\Desktop\陈正扬\处理后数据（含学科）\堆叠面积图.png", dpi=300, bbox_inches="tight")
plt.show()

含学科数据可视化-学科占比环形图

In [ ]:
# ====================== 含学科数据：学科占比环形图（标签带百分比） ======================
print("=== 绘制学科占比环形图（标签带百分比） ===")
df_with_subject = pd.read_excel(FILE_WITH_SUBJECT)
subject_counts = df_with_subject.groupby("学科门类")["项目名称"].count().sort_values(ascending=False)

# 过滤小占比学科
total = subject_counts.sum()
threshold = 0.02
main_subjects = subject_counts[subject_counts / total >= threshold]
other_count = subject_counts[subject_counts / total < threshold].sum()
if other_count > 0:
    other_series = pd.Series({"其他学科": other_count}, index=["其他学科"])
    main_subjects = pd.concat([main_subjects, other_series])

# 构造“学科名称(占比%)”标签
main_subjects_pct = main_subjects / total * 100
labels = [f"{name} ({pct:.1f}%)" for name, pct in zip(main_subjects.index, main_subjects_pct)]

# 绘制环形图（调整变量解压，去掉autotexts）
plt.figure(figsize=(12, 10))
wedges, texts = plt.pie(  # 仅解压2个变量
    main_subjects.values,
    labels=labels,
    autopct=None,
    startangle=90,
    wedgeprops=dict(width=0.3)
)

# 美化标签文字
for text in texts:
    text.set_fontsize(9)

plt.title("2020-2025年各学科门类项目数量占比", fontsize=14, pad=20)
plt.tight_layout()
plt.savefig(r"C:\Users\33759\Desktop\陈正扬\处理后数据（含学科）\学科占比环形图（标签带百分比）.png", dpi=300, bbox_inches="tight")
plt.show()

含学科数据可视化-学科x项目类别热力图

In [ ]:
# ====================== 含学科数据：学科门类和项目类别热力图 ======================
print("=== 绘制学科×项目类别热力图 ===")
# 数据预处理（补充项目类别非空）
df_heatmap = df_with_subject.dropna(subset=["项目类别"])
# 统计交叉计数
heatmap_data = pd.crosstab(df_heatmap["学科门类"], df_heatmap["项目类别"])

# 绘制热力图
plt.figure(figsize=(14, 8))
sns.heatmap(
    heatmap_data,
    annot=True,  # 显示数值
    fmt="d",  # 数值格式为整数
    cmap="YlGnBu",  # 配色
    linewidths=0.5  # 网格线宽度
)
plt.title("学科门类×项目类别热力图", fontsize=14, pad=20)
plt.xlabel("项目类别", fontsize=12)
plt.ylabel("学科门类", fontsize=12)
plt.xticks(rotation=45, ha="right")  # x轴标签旋转，避免重叠
plt.tight_layout()
# 保存图片
plt.savefig(r"C:\Users\33759\Desktop\陈正扬\处理后数据（含学科）\学科项目类别热力图.png", dpi=300, bbox_inches="tight")
plt.show()

（不含学科）数据可视化-TOP20高校条形图

In [ ]:
# ====================== 不含学科数据：立项项目TOP20高校条形图（修正Seaborn警告） ======================
print("=== 绘制TOP20高校条形图 ===")
df_without_subject = pd.read_excel(FILE_WITHOUT_SUBJECT)
df_without_subject = df_without_subject.dropna(subset=["学校名称", "项目名称"])
df_without_subject["学校名称"] = df_without_subject["学校名称"].str.strip().replace(r"[^\u4e00-\u9fa5a-zA-Z0-9]", "", regex=True)

top20_data = df_without_subject.groupby("学校名称")["项目名称"].count().sort_values(ascending=False).head(20)

plt.figure(figsize=(12, 8))
# 修正：添加hue参数并关闭图例
sns.barplot(
    x=top20_data.values,
    y=top20_data.index,
    hue=top20_data.index,  # 将y变量赋值给hue
    palette="viridis",
    legend=False  # 关闭图例
)
plt.title("2020-2025年立项项目TOP20高校", fontsize=14, pad=20)
plt.xlabel("立项数量", fontsize=12)
plt.ylabel("高校名称", fontsize=12)
for i, v in enumerate(top20_data.values):
    plt.text(v + 1, i, str(v), va="center", fontsize=9)
plt.tight_layout()
plt.savefig(r"C:\Users\33759\Desktop\陈正扬\处理后数据（不含学科）\TOP20高校条形图.png", dpi=300, bbox_inches="tight")
plt.show()

（不含学科）数据可视化-TOP20高校中985占比饼图

In [ ]:
# ====================== 不含学科数据：前20高校中985高校占比 ======================
print("=== 绘制TOP20高校985占比饼图 ===")
# 判断TOP20高校是否为985
top20_unis = top20_data.index.tolist()
is_985 = [1 if uni in TOP985_UNIVERSITIES else 0 for uni in top20_unis]
# 统计数量
count_985 = sum(is_985)
count_non985 = len(top20_unis) - count_985
pie_data = [count_985, count_non985]
labels = [f"985高校（{count_985}所）", f"非985高校（{count_non985}所）"]

# 绘制饼图
plt.figure(figsize=(8, 8))
plt.pie(
    pie_data,
    labels=labels,
    autopct="%1.1f%%",
    startangle=90,
    colors=["#FF6B6B", "#4ECDC4"]
)
plt.title("立项TOP20高校中985高校占比", fontsize=14, pad=20)
plt.tight_layout()
# 保存图片
plt.savefig(r"C:\Users\33759\Desktop\陈正扬\处理后数据（不含学科）\TOP20高校985占比.png", dpi=300, bbox_inches="tight")
plt.show()

print("=== 所有可视化图表绘制完成！ ===")